<a href="https://colab.research.google.com/github/divya-parmar5002/DivyaParmar/blob/main/Time_Series_Analysis_with_Cryptocurrency.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df=pd.read_csv("/content/BTC_USD.csv")

In [ ]:
df.head()

,Price,Close,High,Low,Open,Volume
0,Ticker,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
1,Date,NaN,NaN,NaN,NaN,NaN
2,2014-09-17,457.3340148925781,468.17401123046875,452.4219970703125,465.864013671875,21056800
3,2014-09-18,424.44000244140625,456.8599853515625,413.10400390625,456.8599853515625,34483200
4,2014-09-19,394.7959899902344,427.8349914550781,384.5320129394531,424.1029968261719,37919700


In [ ]:
#Removing unwanted rows
df = df.iloc[2:].copy()
#Renaming columns properly
df.columns=['Date','Close','High','Low','Open','Volumn']
#Converting date column
df['Date']=pd.to_datetime(df["Date"])
#converting numeric columns
for col in ['Close','High','Low','Open','Volumn']:
  df[col]=pd.to_numeric(df[col])

#se Date as index
df.set_index("Date",inplace=True)

In [ ]:
df.head()

,Close,High,Low,Open,Volumn
Date,,,,,
2014-09-17,457.334015,468.174011,452.421997,465.864014,21056800
2014-09-18,424.440002,456.859985,413.104004,456.859985,34483200
2014-09-19,394.795990,427.834991,384.532013,424.102997,37919700
2014-09-20,408.903992,423.295990,389.882996,394.673004,36863600
2014-09-21,398.821014,412.425995,393.181000,408.084991,26580100


Data Proprocessing

In [ ]:
df = df.sort_index()


In [ ]:
df.isnull().sum()


,0
Close,0
High,0
Low,0
Open,0
Volumn,0


In [ ]:
df.shape

(4148, 5)

In [ ]:
df=df.dropna()

Exploratory Data Analysis

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

Dashboard 1: Overview/Executive Summary

In [ ]:
#This chart shows the long-term price evolution of Bitcoin, highlighting major bull and bear cycles.
#Bitcoin price trendch
fig=px.line(df,x=df.index,y='Close',title="Bitcoin Closing Price Over Time"
)
fig.show()


In [ ]:
#Volume Trend
fig=px.line(
    df,
    x=df.index,
    y='Volumn',
    title="Bitcoin Trading Volumn Over Time"
)
fig.show()

Dashboard 2: Price Explorer & Candlesticks

In [ ]:
fig=go.Figure(data=[go.Candlestick(
                    x=df.index,
                    open=df['Open'],
                    high=df['High'],
                    low=df['Low'],
                            close=df['Close'])])
fig.update_layout(title="bitcoin Candlestick Chart")
fig.show()

In [ ]:
#Moving Averages(Trend)
df["MA30"]=df["Close"].rolling(30).mean()
df["MA90"]=df["Close"].rolling(90).mean()
fig=px.line(df,
            x=df.index,
            y=["Close","MA30","MA90"],
            title="Bitcoin Price with Moving Averages")

fig.show()

Dashboard 3:Volatility & Risk Analysis

In [ ]:
#Daily Returns
df["Returns"] = df["Close"].pct_change()

fig = px.line(
    df,
    x=df.index,
    y="Returns",
    title="Daily Returns of Bitcoin"
)
fig.show()


In [ ]:
#Rolling Volatility
df["Volatility"] = df["Returns"].rolling(30).std()

fig = px.line(
    df,
    x=df.index,
    y="Volatility",
    title="30-Day Rolling Volatility"
)
fig.show()


Dashboard 4: Forecast & Uncertainty(ARIMA)

In [ ]:
#Train-Test Split
train=df["Close"][:"2024"]
test=df["Close"]["2025":]


In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA

In [ ]:
result = adfuller(df["Close"].dropna())
print("ADF Statistic:", result[0])
print("p-value:", result[1])


ADF Statistic: -0.7161275265839288
p-value: 0.8425278031469005


In [ ]:
#ARIMA Model
model=ARIMA(train,order=(1,1,1))
model_fit=model.fit()
forecast=model_fit.forecast(steps=len(test))

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

No frequency information was provided, so inferred frequency D will be used.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

No frequency information was provided, so inferred frequency D will be used.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

No frequency information was provided, so inferred frequency D will be used.



In [ ]:
#Actual vs Forest
fig = go.Figure()

fig.add_trace(go.Scatter(x=train.index, y=train, name="Train"))
fig.add_trace(go.Scatter(x=test.index, y=test, name="Actual"))
fig.add_trace(go.Scatter(x=test.index, y=forecast, name="Forecast"))

fig.update_layout(title="ARIMA Forecast vs Actual")
fig.show()


In [ ]:
from sklearn.metrics import mean_absolute_error,mean_squared_error
import math

In [ ]:
#Error Metrics
mae = mean_absolute_error(test, forecast)
rmse = math.sqrt(mean_squared_error(test, forecast))

metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE"],
    "Value": [mae, rmse]
})

fig = px.bar(metrics, x="Metric", y="Value", title="Model Evaluation Metrics")
fig.show()


In [ ]:
future = model_fit.forecast(steps=365)

future_dates = pd.date_range(
    start=df.index[-1],
    periods=365,
    freq="D"
)

fig = go.Figure()

fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Historical"))
fig.add_trace(go.Scatter(x=future_dates, y=future, name="Future Forecast"))

fig.update_layout(title="Bitcoin Price Forecast for 2026")
fig.show()


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
sarima = SARIMAX(train,
                 order=(1,1,1),
                 seasonal_order=(1,1,1,365))
sarima_fit = sarima.fit()


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

No frequency information was provided, so inferred frequency D will be used.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

No frequency information was provided, so inferred frequency D will be used.



In [ ]:
prophet_df = df.reset_index()[["Date", "Close"]]
prophet_df.columns = ["ds", "y"]

model = Prophet()
model.fit(prophet_df)

future = model.make_future_dataframe(periods=365)
forecast_prophet = model.predict(future)

model.plot(forecast_prophet)


In [ ]:
df.to_csv("btc_cleaned.csv")

pd.DataFrame({
    "Date": test.index,
    "Actual": test.values,
    "ARIMA_Forecast": forecast.values
}).to_csv("forecast_vs_actual.csv", index=False)
